In [ ]:
!pip install -q transformers accelerate bitsandbytes
!pip install -q transformers accelerate bitsandbytes datasets --upgrade
!pip install -q huggingface_hub

## Imports

In [ ]:
import torch
import pandas as pd
import numpy as np
import re
import json
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##  Configuration

In [ ]:
TEST_FILE = "/path/to/annotation_samples_150.csv"
OUTPUT_DIR = "/path/to/output"

# Model configuration
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
TEMPERATURE = 0.1  # Low temperature for consistency
MAX_NEW_TOKENS = 256

print(f"\nConfiguration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Test file: {TEST_FILE}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Max tokens: {MAX_NEW_TOKENS}")

In [ ]:
from huggingface_hub import login
login()

## Load Model (4-Bit Quantization)


In [ ]:
# Configure 4-bit quantization for efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("✓ Model loaded successfully")
print(f"  Device: {model.device}")
print(f"  Memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## Prepare Few-Shot Examples

In [ ]:
few_shot_examples = [
    {
        'sentence': "(-1 Turn: Speaker 2: Uh, y’know, starve a fever, go to a play for a cold.) (Target Turn: Speaker 4: Joey! Why is Janine not coming over for dinner?!) (+1 Turn: Speaker 2: Well uh, she didn’t want to hang out with you guys two nights in a row. I’m so sorry.)",
        'label': 'negative',
        'reasoning': 'The speaker uses an accusatory tone ("Why is Janine not coming") and an exclamation to demand an explanation for a cancelled social event, indicating frustration.'
    },
    {
        'sentence': "(-1 Turn: Speaker 1: You\'ve got to get back out there, it\'s your party.) (Target Turn: Speaker 2: But they\'re so dull, they\'re all opthamologists.) (+1 Turn: Speaker 1: You\'re an opthamologist.)",
        'label': 'negative',
        'reasoning': 'The subject is being dismissive and critical of the target group ("so dull"), creating a negative social atmosphere and rejecting the object\'s encouragement.'
    },
    {
        'sentence': "(-1 Turn: Speaker 2: Well, yeah! But I\'m not gonna take anymore crap. Okay? No more Mrs. Nice Bucket!) (Target Turn: Speaker 3: Yeah, good for you. Y\'know you\'re tough, you lived on the streets.) (+1 Turn: Speaker 2: Yeah, I\'m gonna go back to being Street Phoebe... Street Phoebe really wouldn\'t be friends with you guys.)",
        'label': 'negative',
        'reasoning': 'The subject explicitly signals social distance and exclusion by stating they "wouldn\'t be friends with you guys."'
    },
    {
        'sentence': "(-1 Turn: Speaker 1: Presenting the award for Favorite Supporting Actress is Joey Tribbiani.) (Target Turn: Speaker 2: ...And I’m sure that Jessica would like to thank my parents... She’d also like to thank my friends, Chandler, Monica, Ross, Phoebe, and Rachel who’s sittin’ right there!) (+1 Turn: N/A)",
        'label': 'positive',
        'reasoning': 'The context involves a public acknowledgment of friendship and gratitude during a celebratory event, indicating a strong positive social bond.'
    },
    {
        'sentence': "(-1 Turn: N/A) (Target Turn: Speaker 1: Goodnight.) (+1 Turn: Speaker 2: Um, thank you for the gift.)",
        'label': 'positive',
        'reasoning': 'The subject expresses direct gratitude ("thank you") for a gift received from the object, which is a clear indicator of a positive social interaction.'
    },
    {
        'sentence': "(-1 Turn: Speaker 1: Yeah, she never misses these conferences! No, I just saw Dr. Kenneth Schwartz!) (Target Turn: Speaker 3: Oh my God! Did you talk to him?) (+1 Turn: Speaker 1: Yeah... what am I going to say?)",
        'label': 'positive',
        'reasoning': 'The use of "Oh my God" and the eager inquiry indicates high excitement, admiration, or positive interest in the target.'
    },
    {
        'sentence': "(-1 Turn: Speaker 2: No.) (Target Turn: Speaker 1: Oh good. Good, look I\'m so sorry, for screwing up that cutting-her-out plan. But I have a new plan. Chandler agreed to call here in a few minutes with an emergency.) (+1 Turn: Speaker 2: Oh! Well, what kind of emergency?)",
        'label': 'neutral',
        'reasoning': 'The subject is mentioned as part of a logistical plan or strategy. The tone is functional and descriptive without expressing personal sentiment.'
    },
    {
        'sentence': "(-1 Turn: N/A) (Target Turn: Speaker 1: Hey.) (+1 Turn: Speaker 2: Hey. Hold on a second. Huh?)",
        'label': 'neutral',
        'reasoning': 'A standard greeting ("Hey") without further emotional modifiers or specific context is a low-intensity, neutral social exchange.'
    },
    {
        'sentence': "(-1 Turn: Speaker 1: Yeah! Sure, sure. So, what was going on with you today?) (Target Turn: Speaker 2: Well, I actually had the weirdest conversation with Joey. He was talking about rules and right and wrong and…) (+1 Turn: Speaker 1: I had the exact same conversation.)",
        'label': 'neutral',
        'reasoning': 'The subject is discussed in terms of the factual content of a conversation rather than being the target of a specific emotional sentiment.'
    },
    {
        'sentence': "(-1 Turn: Speaker 1: So weird to see all these people again... Oh my God, look, there\'s Geoffrey Cleric.) (Target Turn: Speaker 2: Who?) (+1 Turn: Speaker 1: He was roommates with John Rosoff.)",
        'label': 'neutral',
        'reasoning': 'The subject is asking a clarifying factual question ("Who?") to identify a person mentioned, indicating a lack of prior knowledge or specific sentiment.'
    }
]

print(f"✓ Using {len(few_shot_examples)} few-shot examples:")
print(f"  Positive: {sum(1 for ex in few_shot_examples if ex['label'] == 'positive')}")
print(f"  Negative: {sum(1 for ex in few_shot_examples if ex['label'] == 'negative')}")
print(f"  Neutral:  {sum(1 for ex in few_shot_examples if ex['label'] == 'neutral')}")

## Create Prompt Template

In [ ]:
def create_few_shot_prompt(sentence, examples):
    """
    Create a few-shot prompt with clear instructions and examples.

    Args:
        sentence: Test sentence to classify
        examples: List of few-shot examples

    Returns:
        Formatted prompt string
    """
    prompt = """You are an expert at analyzing emotional relationships in social interactions.

Your task: Given a sentence describing an interaction between two people, classify the emotional attribution from the subject toward the object.

Classification Labels:
- positive: favorable emotions (example: love, like, admire, trust, care, respect)
- negative: unfavorable emotions (example: hate, dislike, distrust, anger, contempt)
- neutral: no strong emotion, indifferent, or factual mention

Important Guidelines:
1. Focus on the emotional relationship, not objective facts
2. Consider the intensity and valence of emotions expressed
3. Neutral means lack of emotional charge, not mixed feelings
4. Base your classification only on explicit or strongly implied emotions

Here are some examples:

"""
    # Add examples
    for i, ex in enumerate(examples, 1):
        prompt += f"""Example {i}:
Sentence: "{ex['sentence']}"
Reasoning: {ex['reasoning']}
Classification: {ex['label']}

"""
    # Add query
    prompt += f"""Now classify this new sentence:

Sentence: "{sentence}"

Provide your answer in EXACTLY this format (no other text):
Reasoning: [your step-by-step analysis in one sentence]
Classification: [positive/negative/neutral]
Confidence: [0.7-1.0]
"""
    return prompt

# Test prompt creation
sample_prompt = create_few_shot_prompt("Test sentence", few_shot_examples[:3])
print(f"\n✓ Prompt template created")
print(f"  Prompt length: {len(sample_prompt)} characters")

print('the common enemies of')

## Prediction Function

In [ ]:
def predict_llama3(sentence, examples, model, tokenizer):

    system_prompt = (
        "You are an expert at analyzing emotional relationships in social interactions. "
        "Your task is to classify the emotional attribution from the subject toward the object "
        "as: positive, negative, or neutral. "
        "Provide your answer in the specified format including Reasoning, Classification, and Confidence."
    )

    # 2. Build the message history for Llama 3 Instruct
    messages = [{"role": "system", "content": system_prompt}]

    for ex in examples:
        messages.append({"role": "user", "content": f"Sentence: \"{ex['sentence']}\""})
        assistant_response = (
            f"Reasoning: {ex['reasoning']}\n"
            f"Classification: {ex['label']}\n"
            f"Confidence: {0.9}"
        )
        messages.append({"role": "assistant", "content": assistant_response})

    # Add the current test case
    messages.append({"role": "user", "content": f"Now classify this new sentence: \"{sentence}\""})
    messages.append({"role": "assistant", "content": "Reasoning:"}) # Prime the assistant to start with reasoning

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Get the full generated text from the model
    response_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # Parse response
    label = None
    confidence = 0.8  # Default
    reasoning = ""

    # Extract label (case-insensitive, look for classification line)
    response_lower = response_text.lower()

    # Try to find "classification:" line
    classification_match = re.search(r'classification[:\s]+(positive|negative|neutral)', response_lower)
    if classification_match:
        label = classification_match.group(1)
    else:
        # Fallback: search entire response for keywords if explicit line not found
        if 'positive' in response_lower:
            label = 'positive'
        elif 'negative' in response_lower:
            label = 'negative'
        else:
            label = 'neutral'

    # Extract confidence
    conf_match = re.search(r'confidence[:\s]+([0-9.]+)', response_lower)
    if conf_match:
        try:
            confidence = float(conf_match.group(1))
            # Ensure confidence is in valid range
            confidence = max(0.0, min(1.0, confidence))
        except ValueError:
            confidence = 0.8 # Fallback if not a valid float

    # Extract reasoning
    reason_match = re.search(r'reasoning[:\s]+(.*?)(?:classification|confidence|$)',
                            response_text, re.IGNORECASE | re.DOTALL)
    if reason_match:
        reasoning = reason_match.group(1).strip()
        # Clean up reasoning (remove newlines, extra spaces)
        reasoning = ' '.join(reasoning.split())
    else:
        # Fallback for reasoning if not explicitly found, take part of the response
        reasoning = response_text.split('\n')[0][:150].strip() if response_text else "No specific reasoning found."

    return {
        'pred_label': label,
        'confidence': confidence,
        'reasoning': reasoning,
        'full_response': response_text
    }

## Load Test Data


In [ ]:
# Load test set
test_df = pd.read_csv(TEST_FILE)

print(f"✓ Loaded {len(test_df)} test samples")

# Check columns
print(f"\nColumns: {test_df.columns.tolist()}")

# Infer gold label column
if 'manual_label' in test_df.columns:
    label_col = 'manual_label'
elif 'gold_label' in test_df.columns:
    label_col = 'gold_label'
elif 'label' in test_df.columns:
    label_col = 'label'
elif 'sentiment' in test_df.columns:
    label_col = 'sentiment'
else:
    print("\n Warning: Cannot find a suitable label column. Assuming 'manual_label' for now. Please check your CSV file.")
    label_col = 'manual_label'

print(f"Using label column: '{label_col}'")

# Label distribution
print(f"\nLabel distribution:")
print(test_df[label_col].value_counts())

# Show sample
print(f"\n Sample test data:")
print(test_df[['sentence', label_col]].head(3))

##  Run Predictions

In [ ]:
print("\n" + "="*70)
print("STEP 6: RUNNING LLAMA-3 PREDICTIONS")
print("="*70)

print(f"\nPredicting {len(test_df)} samples...")
print("This will take approximately 15-30 minutes.\n")

predictions = []
pred_labels = []
confidences = []
reasonings = []

# Progress bar
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Predicting"):
    try:
        # Predict
        result = predict_llama3(
            row['sentence'],
            few_shot_examples,
            model,
            tokenizer
        )

        predictions.append(result)
        pred_labels.append(result['pred_label'])
        confidences.append(result['confidence'])
        reasonings.append(result['reasoning'])

    except Exception as e:
        print(f"\n Error on sample {idx}: {e}")
        # Add default prediction on error
        predictions.append({
            'pred_label': 'neutral',
            'confidence': 0.5,
            'reasoning': f'Error: {str(e)}',
            'full_response': ''
        })
        pred_labels.append('neutral')
        confidences.append(0.5)
        reasonings.append(f'Error: {str(e)}')

    # Print progress every 25 samples
    if (idx + 1) % 25 == 0:
        print(f"\n✓ Processed {idx + 1}/{len(test_df)} samples")
        print(f"  Last prediction: {pred_labels[-1]} (conf: {confidences[-1]:.2f})")

# Add predictions to dataframe
test_df['llama3_pred'] = pred_labels
test_df['llama3_conf'] = confidences
test_df['llama3_reasoning'] = reasonings

## Calculate Metrics

In [ ]:
y_true = test_df[label_col].values
y_pred = test_df['llama3_pred'].values

# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro',
                    labels=['positive', 'negative', 'neutral'])
f1_weighted = f1_score(y_true, y_pred, average='weighted',
                       labels=['positive', 'negative', 'neutral'])

# Per-class metrics
precision, recall, f1_per_class, support = precision_recall_fscore_support(
    y_true, y_pred,
    labels=['positive', 'negative', 'neutral'],
    zero_division=0
)

print(f"\n{'='*70}")
print("OVERALL METRICS")
print(f"{'='*70}")
print(f"\nAccuracy:     {accuracy:.3f}")
print(f"F1 Macro:     {f1_macro:.3f}")
print(f"F1 Weighted:  {f1_weighted:.3f}")

print(f"\n{'='*70}")
print("PER-CLASS METRICS")
print(f"{'='*70}")

for i, label in enumerate(['positive', 'negative', 'neutral']):
    print(f"\n{label.upper()}:")
    print(f"  Precision: {precision[i]:.3f}")
    print(f"  Recall:    {recall[i]:.3f}")
    print(f"  F1:        {f1_per_class[i]:.3f}")
    print(f"  Support:   {support[i]}")

# Detailed classification report
print(f"\n{'='*70}")
print("DETAILED CLASSIFICATION REPORT")
print(f"{'='*70}\n")
print(classification_report(
    y_true, y_pred,
    labels=['positive', 'negative', 'neutral'],
    digits=3,
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=['positive', 'negative', 'neutral'])

print(f"{'='*70}")
print("CONFUSION MATRIX")
print(f"{'='*70}\n")
print("                 Predicted")
print("               Pos    Neg    Neu    Total")
print(f"Gold Pos     {cm[0][0]:5d}  {cm[0][1]:5d}  {cm[0][2]:5d}    {support[0]:5d}")
print(f"     Neg     {cm[1][0]:5d}  {cm[1][1]:5d}  {cm[1][2]:5d}    {support[1]:5d}")
print(f"     Neu     {cm[2][0]:5d}  {cm[2][1]:5d}  {cm[2][2]:5d}    {support[2]:5d}")

# Confidence statistics
print(f"\n{'='*70}")
print("CONFIDENCE STATISTICS")
print(f"{'='*70}\n")
print(f"Mean:   {np.mean(confidences):.3f}")
print(f"Median: {np.median(confidences):.3f}")
print(f"Std:    {np.std(confidences):.3f}")
print(f"Min:    {np.min(confidences):.3f}")
print(f"Max:    {np.max(confidences):.3f}")

# Confidence by correctness
correct_mask = y_true == y_pred
conf_correct = [c for i, c in enumerate(confidences) if correct_mask[i]]
conf_incorrect = [c for i, c in enumerate(confidences) if not correct_mask[i]]

print(f"\nConfidence for correct predictions:   {np.mean(conf_correct):.3f}")
print(f"Confidence for incorrect predictions: {np.mean(conf_incorrect):.3f}")
print(f"Difference: {np.mean(conf_correct) - np.mean(conf_incorrect):.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.utils import resample

def run_bootstrapping_analysis(y_true, y_pred, model_name="Model", n_iterations=1000):
    """
    Calculates 95% Confidence Intervals for Macro F1.
    """
    bootstrapped_f1s = []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print(f"Bootstrapping {model_name}...")
    for i in range(n_iterations):
        # Create a bootstrap sample (resample with replacement)
        indices = resample(np.arange(len(y_true)), replace=True)
        resampled_true = y_true[indices]
        resampled_pred = y_pred[indices]

        # Calculate Macro F1 for this sample
        score = f1_score(resampled_true, resampled_pred, average='macro')
        bootstrapped_f1s.append(score)

    # Calculate Confidence Intervals (2.5th and 97.5th percentiles)
    lower = np.percentile(bootstrapped_f1s, 2.5)
    upper = np.percentile(bootstrapped_f1s, 97.5)
    median = np.percentile(bootstrapped_f1s, 50)

    print(f"{model_name} Results:")
    print(f"  Median F1: {median:.3f}")
    print(f"  95% CI:    [{lower:.3f}, {upper:.3f}]")

    return bootstrapped_f1s, (lower, upper)

bootstrapped_f1s, ci_bounds = run_bootstrapping_analysis(y_true, y_pred, "Fine-tuned RoBERTa-3T")

##  Save Results

In [ ]:
# Save predictions CSV
predictions_file = f"{OUTPUT_DIR}/llama3_predictions_150_manual.csv"
test_df.to_csv(predictions_file, index=False)
print(f"\n✓ Saved predictions: {predictions_file}")

# Save metrics JSON
metrics = {
    'model': MODEL_NAME,
    'test_samples': len(test_df),
    'temperature': TEMPERATURE,
    'max_new_tokens': MAX_NEW_TOKENS,
    'overall': {
        'accuracy': float(accuracy),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted)
    },
    'per_class': {
        'positive': {
            'precision': float(precision[0]),
            'recall': float(recall[0]),
            'f1': float(f1_per_class[0]),
            'support': int(support[0])
        },
        'negative': {
            'precision': float(precision[1]),
            'recall': float(recall[1]),
            'f1': float(f1_per_class[1]),
            'support': int(support[1])
        },
        'neutral': {
            'precision': float(precision[2]),
            'recall': float(recall[2]),
            'f1': float(f1_per_class[2]),
            'support': int(support[2])
        }
    },
    'confidence': {
        'mean': float(np.mean(confidences)),
        'median': float(np.median(confidences)),
        'std': float(np.std(confidences)),
        'correct': float(np.mean(conf_correct)),
        'incorrect': float(np.mean(conf_incorrect))
    },
    'confusion_matrix': cm.tolist()
}

metrics_file = f"{OUTPUT_DIR}/llama3_metrics.json"
with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"✓ Saved metrics: {metrics_file}")

## Analysis

In [ ]:
print("\n" + "="*70)
print("STEP 9: SAMPLE PREDICTIONS ANALYSIS")
print("="*70)

# Correct predictions
correct_df = test_df[test_df[label_col] == test_df['llama3_pred']]
print(f"\n CORRECT PREDICTIONS ({len(correct_df)}/{len(test_df)} = {len(correct_df)/len(test_df)*100:.1f}%)")
print("="*70)

for idx, row in correct_df.sample(min(5, len(correct_df)), random_state=42).iterrows():
    print(f"\nSentence: {row['sentence'][:100]}...")
    print(f"Gold: {row[label_col]}, Predicted: {row['llama3_pred']} ✓")
    print(f"Confidence: {row['llama3_conf']:.2f}")
    print(f"Reasoning: {row['llama3_reasoning'][:150]}...")

# Incorrect predictions
incorrect_df = test_df[test_df[label_col] != test_df['llama3_pred']]
print(f"\n\n❌ INCORRECT PREDICTIONS ({len(incorrect_df)}/{len(test_df)} = {len(incorrect_df)/len(test_df)*100:.1f}%)")
print("="*70)

for idx, row in incorrect_df.sample(min(5, len(incorrect_df)), random_state=42).iterrows():
    print(f"\nSentence: {row['sentence'][:100]}...")
    print(f"Gold: {row[label_col]}, Predicted: {row['llama3_pred']} ✗")
    print(f"Confidence: {row['llama3_conf']:.2f}")
    print(f"Reasoning: {row['llama3_reasoning'][:150]}...")

##  Final Summary

In [ ]:
print("\n" + "="*70)
print("✓ LLAMA-3 EVALUATION COMPLETE!")
print("="*70)

print(f"\n📊 FINAL SUMMARY:")
print(f"  Model:        {MODEL_NAME}")
print(f"  Test samples: {len(test_df)}")
print(f"  Accuracy:     {accuracy:.3f}")
print(f"  F1 Macro:     {f1_macro:.3f}")
print(f"  F1 Positive:  {f1_per_class[0]:.3f}")
print(f"  F1 Negative:  {f1_per_class[1]:.3f}")
print(f"  F1 Neutral:   {f1_per_class[2]:.3f}")

print(f"\n📁 Output files:")
print(f"  1. {predictions_file}")
print(f"  2. {metrics_file}")